# VAR-d20 Smoke Test

This notebook verifies that the copied VAR code can load the VAE and VAR-d20 checkpoint, then generate a small ImageNet-class sample grid.

In [ ]:
import os
import subprocess
from pathlib import Path

REPO_URL = "https://github.com/<your-username>/<your-repo>.git"  # TODO: replace once pushed
BRANCH = "main"
WORKSPACE = Path('/content/VAR_Style_Transfer_Workspace')

if not WORKSPACE.exists():
    assert '<your-username>' not in REPO_URL, 'Set REPO_URL to your GitHub repo first.'
    subprocess.run(['git', 'clone', '--depth', '1', '--branch', BRANCH, REPO_URL, str(WORKSPACE)], check=True)
else:
    subprocess.run(['git', '-C', str(WORKSPACE), 'pull'], check=True)

os.chdir(WORKSPACE / 'VAR')
print(Path.cwd())


In [ ]:
!nvidia-smi
!pip install -q huggingface_hub einops


In [ ]:
from pathlib import Path
from huggingface_hub import hf_hub_download

weights_dir = Path('/content/VAR_weights')
weights_dir.mkdir(parents=True, exist_ok=True)

vae_path = hf_hub_download(
    repo_id='FoundationVision/var',
    filename='vae_ch160v4096z32.pth',
    local_dir=weights_dir,
)
var_path = hf_hub_download(
    repo_id='FoundationVision/var',
    filename='var_d20.pth',
    local_dir=weights_dir,
)

vae_path, var_path


In [ ]:
import random
import torch
from torchvision.utils import make_grid
from PIL import Image
from IPython.display import display
from models import build_vae_var

MODEL_DEPTH = 20
patch_nums = (1, 2, 3, 4, 5, 6, 8, 10, 13, 16)
device = 'cuda' if torch.cuda.is_available() else 'cpu'

vae, var = build_vae_var(
    V=4096,
    Cvae=32,
    ch=160,
    share_quant_resi=4,
    device=device,
    patch_nums=patch_nums,
    num_classes=1000,
    depth=MODEL_DEPTH,
    shared_aln=False,
)

vae.load_state_dict(torch.load(vae_path, map_location='cpu'), strict=True)
var.load_state_dict(torch.load(var_path, map_location='cpu'), strict=True)
vae.eval(); var.eval()
for p in vae.parameters(): p.requires_grad_(False)
for p in var.parameters(): p.requires_grad_(False)

print(f'Loaded VAR-d{MODEL_DEPTH} on {device}')


In [ ]:
torch.manual_seed(0)
B = 4
labels = torch.tensor([281, 285, 292, 207], device=device)  # tabby, Egyptian cat, tiger, golden retriever

with torch.no_grad(), torch.autocast(device_type='cuda', dtype=torch.float16, enabled=(device == 'cuda')):
    recon = var.autoregressive_infer_cfg(
        B=B,
        label_B=labels,
        cfg=4.0,
        top_k=900,
        top_p=0.95,
        g_seed=0,
    )

grid = make_grid(recon, nrow=2, padding=2)
grid = grid.detach().cpu().clamp(0, 1).permute(1, 2, 0).numpy()
img = Image.fromarray((grid * 255).astype('uint8'))
out_dir = Path('/content/VAR_outputs')
out_dir.mkdir(parents=True, exist_ok=True)
out_path = out_dir / 'var_d20_smoke_grid.png'
img.save(out_path)
display(img)
print(out_path)
